# 3. Evaluation: the setting chosen on validation, then the test writers once

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chingkheinganba231005/Meitei-Mayek-Diffusion-model/blob/main/notebooks/3_evaluate.ipynb)

**Runtime: A100.** About 1.5 hours.

The deployed word recogniser (from Hugging Face, its version recorded) reads the generated
words; HWD compares them with the real words of the same writers.

1. **Checks**: the recogniser is fetched, and HWD is checked on real words (a set against
   itself gives 0, other words of the same writers more).
2. **Validation grid (E3)** on 200 validation writers, 12 words each, in stages: the average of
   the weights; guidance and its interval; the sampler and its steps; the number of references.
   Each stage keeps the lowest HWD among the settings whose greedy CER is at most the
   reference CER (the recogniser on the real words) plus 1 point. Resumable: rows already in
   `results/eval_val_grid.json` are kept.
3. **Test (E1, E2)**, once, on all 1,000 test writers with the chosen setting: CER and WER,
   HWD, FID, KID, the wrong-style control (the same texts in the next writer's hand) and the
   reference level, and a sheet of writers (references | real | generated).
4. **Figures** of the trade-offs (E7: evaluations of the network against quality).

Inputs: `WORK/runs/main/final.pt`, `WORK/data/{val,test}`. Outputs: `WORK/results/eval_val_grid.json`,
`WORK/results/eval_test.json`, `WORK/eval/`.

In [ ]:
import os

# Where things are. Change these to match your Drive.
WORK = "/content/drive/MyDrive/meitei-word-diffusion"         # this project's working folder
WORK_OLD = "/content/drive/MyDrive/meitei-word-recognition"   # the word-recognition project's folder
REPO = "chingkheinganba231005/Meitei-Mayek-Diffusion-model"
BRANCH = "main"
WORD_REPO = "chingkheinganba231005/meitei-mayek-word-recognition"
WORD_COMMIT = "bdee9e7"                                       # the synthesiser this project builds on
CONTENT = "/content"                                          # the runtime's own disk

HF_REC = "Chingkheinganba/handwritten-meitei-mayek-word-recognition"   # the deployed word recogniser
REC_REVISION = "main"                       # its version is recorded in results/recogniser_source.json
HWD_COMMIT = "eabb5b5"                      # the HWD package (github.com/aimagelab/HWD)

RUN = "main"
VAL_WRITERS = 200
TEST_WRITERS = 1000

DISCONNECT_AT_END = True    # the last cell disconnects the runtime (Colab bills a connected runtime)

In [ ]:
import json, re, shutil, subprocess, sys, time
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
os.chdir(CONTENT)
CODE, WORD = f"{CONTENT}/code", f"{CONTENT}/word_repo"


def run(*args):
    """Runs a command, shows its output, and stops the notebook if it fails."""
    r = subprocess.run([str(a) for a in args], capture_output=True, text=True)
    if (r.stdout + r.stderr).strip():
        print(r.stdout + r.stderr)
    if r.returncode:
        raise SystemExit(f"exit code {r.returncode}: {' '.join(str(a) for a in args)[:200]}")


def stream(*args):
    """Runs a long command, showing its output as it comes; stops the notebook if it fails.
    Returns the output."""
    p = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end="")
        lines.append(line)
    if p.wait():
        raise SystemExit(f"exit code {p.returncode}: {' '.join(str(a) for a in args)[:200]}")
    return "".join(lines)


def clone(url, path, ref):
    """A fresh copy of a repository at a branch or a commit."""
    shutil.rmtree(path, ignore_errors=True)
    run("git", "clone", "-q", url, path)
    run("git", "-C", path, "checkout", "-q", ref)


clone(f"https://github.com/{REPO}.git", CODE, BRANCH)
clone(f"https://github.com/{WORD_REPO}.git", WORD, WORD_COMMIT)
sys.path[:0] = [CODE, WORD]
os.chdir(CODE)

run(sys.executable, "-m", "pip", "install", "-q", "diffusers==0.41.0", "safetensors", "huggingface_hub", "timm")

if not os.path.exists(f"{CONTENT}/HWD/setup.py"):
    clone("https://github.com/aimagelab/HWD.git", f"{CONTENT}/HWD", HWD_COMMIT)
run(sys.executable, "-m", "pip", "install", "-q", f"{CONTENT}/HWD", "gudhi")   # gudhi: imported by hwd.scores

for d in ("data", "runs", "results", "eval"):
    os.makedirs(f"{WORK}/{d}", exist_ok=True)
run("git", "-C", CODE, "log", "-1", "--format=code: %h %s")

run("nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader")

REC = f"{CONTENT}/recogniser"


def get_recogniser():
    """The deployed recogniser (recogniser.pt, char_lm.pkl, decoding.json) at the version recorded the
    first time (results/recogniser_source.json), so that every notebook scores with the same one."""
    from huggingface_hub import HfApi, snapshot_download
    kept = f"{WORK}/results/recogniser_source.json"
    if os.path.exists(kept):
        source = json.loads(Path(kept).read_text())
    else:
        source = {"repo": HF_REC, "revision": REC_REVISION, "sha": HfApi().model_info(HF_REC, revision=REC_REVISION).sha}
    if not os.path.exists(f"{REC}/source.json"):
        snapshot_download(HF_REC, revision=source["sha"], local_dir=REC,
                          allow_patterns=["recogniser.pt", "char_lm.pkl", "decoding.json", "results.json"])
        Path(f"{REC}/source.json").write_text(json.dumps(source))
    if not os.path.exists(kept):
        shutil.copy(f"{REC}/source.json", kept)
    print("recogniser:", json.loads(Path(f"{REC}/source.json").read_text()))


get_recogniser()

from IPython.display import Image as Show, display
import pandas as pd
WEIGHTS = f"{WORK}/runs/{RUN}/final.pt"
GRID, TEST = f"{WORK}/results/eval_val_grid.json", f"{WORK}/results/eval_test.json"
if not os.path.exists(WEIGHTS):
    raise SystemExit(f"{WEIGHTS} does not exist yet: finish notebook 2 first")

## 1. HWD on real words

The HWD package's own check: the real words of 20 validation writers against themselves (HWD
0), and against other words of the same writers (more than 0).

In [ ]:
import numpy as np
from mayek_diffusion.data import WordSet
from mayek_diffusion.evaluate import write_writer_folders
from mayek_diffusion.scores import StyleScorer

val = WordSet(f"{WORK}/data/val")
check = f"{CONTENT}/eval/hwd_check"
shutil.rmtree(check, ignore_errors=True)
per = val.per_writer
for name, items in (("a", range(4, 10)), ("b", range(10, 16))):
    idx = [w * per + i for w in range(min(20, len(val) // per)) for i in items]
    write_writer_folders([val.target(i).astype(np.float32) / 255 for i in idx], [int(val.writer[i]) for i in idx],
                         f"{check}/{name}")
scorer = StyleScorer()
same, other = scorer(f"{check}/a", f"{check}/a")["hwd"], scorer(f"{check}/b", f"{check}/a")["hwd"]
print(f"HWD of a set with itself {same:.4f}; other words of the same writers {other:.4f}")
if not (same < 1e-6 < other):
    raise SystemExit("HWD does not behave as expected: look at the HWD package before going on")

## 2. Validation grid (E3)

About 20 settings of 2,400 words each. Each row: greedy and language-model CER and WER, HWD,
the network evaluations a word costs, and seconds a word on this GPU.

In [ ]:
stream(sys.executable, "-u", "scripts/evaluate.py", "tune", "--word-repo", WORD, "--weights", WEIGHTS,
       "--data", f"{WORK}/data", "--recogniser", REC, "--writers", VAL_WRITERS, "--work", f"{CONTENT}/eval/val",
       "--out", GRID)
grid = json.loads(Path(GRID).read_text())
rows = [{"key": k, "CER": r["greedy_cer"], "CER LM": r["with_lm_cer"], "WER": r["greedy_wer"], "HWD": r.get("hwd"),
         "reference CER": r["reference_greedy_cer"], "NFE": r["nfe"], "guided": r["nfe_guided"],
         "s/word": r["seconds_per_word"]} for k, r in grid["rows"].items()]
display(pd.DataFrame(rows))
print("chosen:", grid["chosen"])

## 3. Test writers (E1, E2), once

12,000 generated words, and as many again for the wrong-style control. Run once: if
`results/eval_test.json` exists, it is shown instead.

In [ ]:
if not os.path.exists(TEST):
    stream(sys.executable, "-u", "scripts/evaluate.py", "score", "--word-repo", WORD, "--weights", WEIGHTS,
           "--data", f"{WORK}/data", "--split", "test", "--writers", TEST_WRITERS, "--setting", GRID,
           "--recogniser", REC, "--control", "--fid-kid", "--work", f"{CONTENT}/eval/test", "--out", TEST,
           "--sheet", f"{WORK}/eval/test_writers.png", "--label", "test")
t = json.loads(Path(TEST).read_text())
print(json.dumps({k: t.get(k) for k in ("writers", "words", "greedy_cer", "greedy_wer", "with_lm_cer", "with_lm_wer",
                                       "reference_greedy_cer", "reference_with_lm_cer", "hwd", "fid", "kid")}, indent=1))
print("wrong-style control:", {k: t["control"].get(k) for k in ("hwd", "greedy_cer")})
display(Show(f"{WORK}/eval/test_writers.png"))

## 4. Figures: quality against cost (E7)

In [ ]:
import matplotlib.pyplot as plt

grid = json.loads(Path(GRID).read_text())
os.makedirs(f"{WORK}/eval/figures", exist_ok=True)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
for name, marker in (("sampler", "o"), ("guidance", "s")):
    rs = [grid["rows"][k] for k in grid["stages"][name]["keys"]]
    cost = [r["nfe"] + r["nfe_guided"] for r in rs]
    ax[0].scatter(cost, [100 * r["greedy_cer"] for r in rs], marker=marker, label=name)
    ax[1].scatter(cost, [r.get("hwd") or float("nan") for r in rs], marker=marker, label=name)
ax[0].axhline(100 * grid["stages"]["sampler"]["reference_greedy_cer"], color="grey", ls="--", lw=1)
ax[0].set_ylabel("greedy CER (%)"); ax[1].set_ylabel("HWD")
for a in ax:
    a.set_xlabel("network evaluations a word"); a.legend()
fig.tight_layout()
fig.savefig(f"{WORK}/eval/figures/cost_quality.png", dpi=150)
plt.show()

## Disconnect

Colab bills a connected runtime whether or not it is working. The cell below disconnects it
(`DISCONNECT_AT_END = True`); otherwise use *Runtime > Disconnect and delete runtime*. Every
result is already on Drive.

In [ ]:
# Colab bills a connected runtime whether or not it is working: disconnect it now.
if DISCONNECT_AT_END:
    from google.colab import runtime
    runtime.unassign()
else:
    print("Finished. Disconnect the runtime: Runtime > Disconnect and delete runtime.")